# Comparison Report Generator

Aggregates the `gold_comparison_<model>.csv` files produced by `LLM_evaluate_model.ipynb` into 5 summary charts. Reads only already-computed local files — makes no API calls.

Run all cells top to bottom.

## Configuration

In [1]:
# Configuration

from pathlib import Path

PROJECT_ROOT = Path("/Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT")
RESULTS_ROOT = PROJECT_ROOT / "results"
OUT_DIRS = [
    PROJECT_ROOT / "figures" / "model_comparison_report",
    PROJECT_ROOT / "submission_results" / "summary_charts",
]
for d in OUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

MODELS = ["sonnet4_6", "gpt5_2", "gpt4o"]
MODEL_DISPLAY = {"sonnet4_6": "Sonnet 4.6", "gpt5_2": "GPT-5.2", "gpt4o": "GPT-4o"}
COLOR = {"gpt4o": "#eda100", "gpt5_2": "#1baf7a", "sonnet4_6": "#2a78d6", "tie": "#b3ada2"}
TOPDOWN = ["gpt4o", "gpt5_2", "sonnet4_6"]        # bar draw order, top -> bottom
LEGEND_ORDER = ["sonnet4_6", "gpt5_2", "gpt4o"]   # legend order, left -> right

STUDENT_DISPLAY = {"taylor": "Taylor Swift", "dapaw": "DaPaw", "rose": "Rose", "sj3747": "SJ3747"}
STUDENT_ORDER = ["taylor", "dapaw", "rose", "sj3747"]
BEHAVIOR_ORDER = ["enacting", "planning", "reflecting", "monitoring", "interacting"]

print("Configuration loaded.")


Configuration loaded.


## Collect Comparisons

In [2]:
# Collect gold_comparison_<model>.csv across all combinations
#
# Each row already carries a corrected_category column (see
# LLM_evaluate_model.ipynb): for ai_correct=TRUE gold points this equals the
# raw boundary-classification category; for ai_correct=FALSE points category
# 1 and 10 are swapped, since those points are human-validated false
# positives and avoiding them, not matching them, is success.

import csv
import glob
from collections import defaultdict, Counter


def collect_comparisons():
    """Returns:
      overall[model]            -> Counter(corrected_category)
      by_student[student][model] -> Counter(corrected_category)
      by_behavior[behavior][model] -> Counter(corrected_category)
      combo_exact[(student,day,behavior)][model] -> exact-match % (corrected)
    """
    overall = defaultdict(Counter)
    by_student = defaultdict(lambda: defaultdict(Counter))
    by_behavior = defaultdict(lambda: defaultdict(Counter))
    combo_rows = defaultdict(lambda: defaultdict(list))

    files = sorted(glob.glob(str(RESULTS_ROOT / "*" / "*" / "*" / "gold_comparison_*.csv")))
    for fp in files:
        parts = Path(fp).relative_to(RESULTS_ROOT).parts
        student, day, behavior = parts[0], parts[1], parts[2]
        model = None
        for m in MODELS:
            if fp.endswith(f"gold_comparison_{m}.csv"):
                model = m
                break
        if model is None:
            continue

        with open(fp, newline="", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))
        if not rows:
            continue

        cc_counter = Counter(r["corrected_category"] for r in rows)
        overall[model].update(cc_counter)
        by_student[student][model].update(cc_counter)
        by_behavior[behavior][model].update(cc_counter)
        combo_rows[(student, day, behavior)][model] = rows

    combo_exact = {}
    for combo, model_rows in combo_rows.items():
        combo_exact[combo] = {}
        for model, rows in model_rows.items():
            n = len(rows)
            exact = sum(1 for r in rows if r["corrected_category"] == "1")
            combo_exact[combo][model] = 100 * exact / n if n else 0.0

    return overall, by_student, by_behavior, combo_exact


overall, by_student, by_behavior, combo_exact = collect_comparisons()
print(f"Collected comparisons for {len(combo_exact)} combinations.")


Collected comparisons for 33 combinations.


## Aggregate

In [3]:
# Aggregate: headline exact/miss rates, by-student, by-behavior, combo wins

def pct_table(counter_by_model, cat):
    out = {}
    for model in MODELS:
        total = sum(counter_by_model[model].values())
        cnt = counter_by_model[model].get(cat, 0)
        out[model] = 100 * cnt / total if total else 0.0
    return out


headline_exact = pct_table(overall, "1")
headline_miss = pct_table(overall, "10")

by_student_exact = {s: pct_table(by_student[s], "1") for s in by_student}
by_behavior_exact = {b: pct_table(by_behavior[b], "1") for b in by_behavior}

wins = Counter()
ties = 0
for combo, model_pcts in combo_exact.items():
    vals = {m: model_pcts.get(m, 0.0) for m in MODELS}
    maxv = max(vals.values())
    winners = [m for m, v in vals.items() if abs(v - maxv) < 1e-9]
    if len(winners) == 1:
        wins[winners[0]] += 1
    else:
        ties += 1
total_combos = len(combo_exact)

print("Headline exact-match %:", {MODEL_DISPLAY[m]: round(headline_exact[m], 1) for m in MODELS})
print("Headline complete-miss %:", {MODEL_DISPLAY[m]: round(headline_miss[m], 1) for m in MODELS})
print("Combo wins:", {MODEL_DISPLAY[m]: wins[m] for m in MODELS}, "ties:", ties, "/", total_combos)


Headline exact-match %: {'Sonnet 4.6': 58.7, 'GPT-5.2': 46.0, 'GPT-4o': 56.8}
Headline complete-miss %: {'Sonnet 4.6': 16.6, 'GPT-5.2': 13.5, 'GPT-4o': 9.6}
Combo wins: {'Sonnet 4.6': 12, 'GPT-5.2': 7, 'GPT-4o': 7} ties: 7 / 33


## Chart Helpers

In [4]:
# Chart helpers

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#333333"


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.grid(axis="x", color="#e6e6e6", linewidth=1, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0, labelsize=15, colors="#222222")
    ax.tick_params(axis="x", labelsize=12, colors="#444444")


def add_title(fig, title, subtitle=None):
    fig.text(0.02, 0.96, title, fontsize=21, fontweight="bold", color="#111111", va="top")
    if subtitle:
        fig.text(0.02, 0.86, subtitle, fontsize=13, color="#555555", va="top")


def grouped_hbar(ax, group_labels, group_values, bar_h=0.24, gap=0.06, fmt="{:.1f}%"):
    n_groups = len(group_labels)
    n_bars = len(TOPDOWN)
    y_group_centers = list(range(n_groups))[::-1]
    yticks = []
    for gi, (label, vals) in enumerate(zip(group_labels, group_values)):
        center = y_group_centers[gi]
        for bi, model in enumerate(TOPDOWN):
            y = center + (n_bars - 1) / 2 * (bar_h + gap) - bi * (bar_h + gap)
            v = vals.get(model, 0.0)
            ax.barh(y, v, height=bar_h, color=COLOR[model], zorder=3)
            ax.text(v + max(vals.values()) * 0.012 + 0.15, y, fmt.format(v),
                     va="center", ha="left", fontsize=11.5, fontweight="bold", color="#111111")
        yticks.append(center)
    ax.set_yticks(yticks)
    ax.set_yticklabels(group_labels, fontsize=15)
    ax.set_ylim(-0.6, n_groups - 1 + 0.6)


def save(fig, name):
    for d in OUT_DIRS:
        fig.savefig(d / name, dpi=150, bbox_inches="tight", facecolor="white")


def legend(ax, y_offset):
    handles = [plt.Rectangle((0, 0), 1, 1, color=COLOR[m]) for m in LEGEND_ORDER]
    ax.legend(handles, [MODEL_DISPLAY[m] for m in LEGEND_ORDER], loc="upper center",
              bbox_to_anchor=(0.5, y_offset), ncol=3, frameon=False, fontsize=13)


print("Chart helpers defined.")


Chart helpers defined.


## Chart 1 — Headline Exact Match vs. Complete Miss

In [5]:
# Chart 1 — headline exact vs. miss

fig, ax = plt.subplots(figsize=(11, 4.6))
fig.subplots_adjust(top=0.62, bottom=0.16, left=0.14, right=0.95)
group_labels = ["Exact match rate", "Complete miss rate"]
group_values = [headline_exact, headline_miss]
grouped_hbar(ax, group_labels, group_values)
style_axes(ax)
ax.set_xlim(0, max(max(v.values()) for v in group_values) * 1.22)
ax.set_xlabel("%", fontsize=13, color="#444444")
legend(ax, -0.28)
add_title(fig, "Exact Match vs. Complete Miss (corrected — aggregated across all gold-covered combinations)",
          f"{sum(overall['sonnet4_6'].values())} gold-point comparisons per model  |  "
          f"ai_correct=FALSE points scored with inverted (avoidance) criterion")
save(fig, "01_headline_exact_miss.png")
plt.close(fig)
print("Saved 01_headline_exact_miss.png")


Saved 01_headline_exact_miss.png


## Chart 2 — Combo-Level Wins

In [6]:
# Chart 2 — combo-level wins

fig, ax = plt.subplots(figsize=(12, 5.2))
fig.subplots_adjust(top=0.72, bottom=0.14, left=0.16, right=0.95)
labels = ["Sonnet 4.6", "GPT-5.2", "GPT-4o", "Ties"]
values = [wins.get("sonnet4_6", 0), wins.get("gpt5_2", 0), wins.get("gpt4o", 0), ties]
colors = [COLOR["sonnet4_6"], COLOR["gpt5_2"], COLOR["gpt4o"], COLOR["tie"]]
y = list(range(len(labels)))[::-1]
for yi, val, col in zip(y, values, colors):
    ax.barh(yi, val, height=0.55, color=col, zorder=3)
    ax.text(val + total_combos * 0.012, yi, f"{val} / {total_combos}", va="center", ha="left",
             fontsize=15, fontweight="bold", color="#111111")
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=17)
ax.set_ylim(-0.6, len(labels) - 1 + 0.6)
style_axes(ax)
ax.set_xlim(0, max(values) * 1.3)
ax.set_xlabel(f"Combinations won (out of {total_combos} with gold data)", fontsize=13, color="#444444")
add_title(fig, "Combo-Level Wins (corrected): Highest Exact-Match % per Combination",
          "Each gold-covered combination counted once, regardless of gold-point volume")
save(fig, "02_combo_wins.png")
plt.close(fig)
print("Saved 02_combo_wins.png")


Saved 02_combo_wins.png


## Chart 3 — Exact-Match Rate by Behavior

In [7]:
# Chart 3 — exact-match rate by behavior

group_labels = [b for b in BEHAVIOR_ORDER if b in by_behavior_exact]
group_values = [by_behavior_exact[b] for b in group_labels]
fig, ax = plt.subplots(figsize=(11, 6.4))
fig.subplots_adjust(top=0.80, bottom=0.10, left=0.16, right=0.95)
grouped_hbar(ax, group_labels, group_values)
style_axes(ax)
ax.set_xlim(0, max(max(v.values()) for v in group_values) * 1.2)
ax.set_xlabel("%", fontsize=13, color="#444444")
legend(ax, -0.14)
add_title(fig, "Exact-Match Rate by Behavior Type (corrected)",
          "No single model wins every behavior category")
save(fig, "03_by_behavior.png")
plt.close(fig)
print("Saved 03_by_behavior.png")


Saved 03_by_behavior.png


## Chart 4 — Exact-Match Rate by Student

In [8]:
# Chart 4 — exact-match rate by student

group_labels = [STUDENT_DISPLAY[s] for s in STUDENT_ORDER if s in by_student_exact]
group_values = [by_student_exact[s] for s in STUDENT_ORDER if s in by_student_exact]
fig, ax = plt.subplots(figsize=(11, 5.6))
fig.subplots_adjust(top=0.76, bottom=0.11, left=0.16, right=0.95)
grouped_hbar(ax, group_labels, group_values)
style_axes(ax)
ax.set_xlim(0, max(max(v.values()) for v in group_values) * 1.22)
ax.set_xlabel("%", fontsize=13, color="#444444")
legend(ax, -0.16)
add_title(fig, "Exact-Match Rate by Student (corrected)",
          "Differences track how much speech/gesture data each student's sessions contain")
save(fig, "04_by_student.png")
plt.close(fig)
print("Saved 04_by_student.png")


Saved 04_by_student.png


## Chart 5 — Full 10-Category Boundary Classification

In [9]:
# Chart 5 — full 10-category boundary classification

CAT_LABELS = {
    "1": "1 Exact Equal", "2": "2 Start<Gold/End=", "3": "3 Start=/End>Gold",
    "4": "4 Start</End inside", "5": "5 Start inside/End>", "6": "6 Start=/End<Gold",
    "7": "7 Start>Gold/End=", "8": "8 AI inside Gold", "9": "9 AI contains Gold",
    "10": "10 Complete Miss",
}
cats = [str(i) for i in range(1, 11)]
full_breakdown = {c: pct_table(overall, c) for c in cats}
group_labels = [CAT_LABELS[c] for c in cats]
group_values = [full_breakdown[c] for c in cats]

fig, ax = plt.subplots(figsize=(11, 11.6))
fig.subplots_adjust(top=0.83, bottom=0.06, left=0.20, right=0.95)
grouped_hbar(ax, group_labels, group_values, bar_h=0.24, gap=0.05)
style_axes(ax)
ax.set_xlim(0, max(max(v.values()) for v in group_values) * 1.28)
ax.set_xlabel("%", fontsize=13, color="#444444")
legend(ax, -0.045)
add_title(fig, "Full 10-Category Boundary Classification (corrected, aggregated %)",
          "Cat 1 = exact match, Cat 10 = no overlap, 2-9 = partial (FALSE points: 1 and 10 swapped)")
save(fig, "05_full_category_breakdown.png")
plt.close(fig)
print("Saved 05_full_category_breakdown.png")


Saved 05_full_category_breakdown.png


## Done

In [10]:
# Done

print("All 5 charts written to:")
for d in OUT_DIRS:
    print(" ", d)


All 5 charts written to:
  /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/figures/model_comparison_report
  /Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT/submission_results/summary_charts
